# PVOD Dataset Exploration
10 stations, 15-min resolution, 2018-2022.  
Columns: 7 NWP + 6 LMD + `power` + `is_day`.

In [ ]:
import sys
sys.path.insert(0, '..')

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
DATA_DIR = Path('../data/processed')
STATIONS = sorted([p.stem for p in DATA_DIR.glob('station*.csv')])
print('Stations:', STATIONS)

## 1. Load all stations

In [ ]:
dfs = {}
for sid in STATIONS:
    df = pd.read_csv(DATA_DIR / f'{sid}.csv', parse_dates=['date_time'], index_col='date_time')
    dfs[sid] = df

NWP_COLS = [c for c in dfs[STATIONS[0]].columns if c.startswith('nwp_')]
LMD_COLS = [c for c in dfs[STATIONS[0]].columns if c.startswith('lmd_')]
print(f'NWP cols ({len(NWP_COLS)}):', NWP_COLS)
print(f'LMD cols ({len(LMD_COLS)}):', LMD_COLS)
dfs[STATIONS[0]].head(3)

## 2. Basic statistics per station

In [ ]:
stats = []
for sid, df in dfs.items():
    day = df[df['is_day'] == 1]
    stats.append({
        'station': sid,
        'total_steps': len(df),
        'daytime_steps': len(day),
        'date_start': df.index.min().date(),
        'date_end': df.index.max().date(),
        'power_max': df['power'].max(),
        'power_mean_day': day['power'].mean(),
        'missing_pct': df['power'].isna().mean() * 100,
    })
stats_df = pd.DataFrame(stats).set_index('station')
stats_df

## 3. Train / Val / Test split overview

In [ ]:
# Default split: 70/10/20 by time for each station
split_rows = []
for sid, df in dfs.items():
    n = len(df)
    n_tr = int(n * 0.7)
    n_va = int(n * 0.1)
    split_rows.append({
        'station': sid,
        'train_end': df.index[n_tr - 1].date(),
        'val_end': df.index[n_tr + n_va - 1].date(),
        'test_end': df.index[-1].date(),
        'train_days': n_tr // 96,
        'val_days': n_va // 96,
        'test_days': (n - n_tr - n_va) // 96,
    })
pd.DataFrame(split_rows).set_index('station')

## 4. Power distribution across stations

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharey=False)
for ax, sid in zip(axes.flat, STATIONS):
    day_power = dfs[sid][dfs[sid]['is_day'] == 1]['power']
    ax.hist(day_power, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(sid, fontsize=10)
    ax.set_xlabel('Power')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
fig.suptitle('Daytime Power Distribution (10 Stations)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. NWP vs LMD irradiance correlation

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, sid in zip(axes.flat, STATIONS):
    df = dfs[sid]
    day = df[df['is_day'] == 1]
    ax.scatter(day['nwp_globalirrad'], day['lmd_totalirrad'],
               s=1, alpha=0.2, color='darkorange')
    ax.set_title(sid, fontsize=10)
    ax.set_xlabel('NWP global irrad')
    ax.set_ylabel('LMD total irrad')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
fig.suptitle('NWP vs LMD Irradiance (daytime)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Seasonal power profile (mean intraday curve by quarter)

In [ ]:
sid = STATIONS[0]
df = dfs[sid].copy()
df['quarter'] = df.index.quarter
df['time_slot'] = df.index.hour * 4 + df.index.minute // 15

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
labels = ['Q1 (Jan-Mar)', 'Q2 (Apr-Jun)', 'Q3 (Jul-Sep)', 'Q4 (Oct-Dec)']
for q, color, label in zip([1, 2, 3, 4], colors, labels):
    sub = df[df['quarter'] == q]
    profile = sub.groupby('time_slot')['power'].mean()
    ax.plot(profile.index / 4, profile.values, color=color, label=label, lw=1.5)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Mean power')
ax.set_title(f'{sid} — Seasonal intraday power profile')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Feature correlation heatmap (station00, daytime)

In [ ]:
df = dfs[STATIONS[0]]
day = df[df['is_day'] == 1]
feat_cols = NWP_COLS + LMD_COLS + ['power']
corr = day[feat_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.4, ax=ax, annot_kws={'size': 8})
ax.set_title(f'{STATIONS[0]} — Feature correlation (daytime)')
plt.tight_layout()
plt.show()

## 8. FCM weather regime distribution

In [ ]:
# Run FCM on daily irradiance profiles to visualize regime membership
from src.utils.fcm import FCM  # adjust import if path differs

K = 3
sid = STATIONS[0]
df = dfs[sid].copy()
# Build daily irradiance profile matrix (daytime slots only, 96 steps/day)
irrad = df['nwp_globalirrad'].values.reshape(-1, 96)  # (n_days, 96)
fcm = FCM(n_clusters=K, m=2.0, max_iter=150)
fcm.fit(irrad)
U = fcm.u  # (K, n_days) membership

fig, axes = plt.subplots(1, K, figsize=(14, 4), sharey=True)
slots = np.arange(96) / 4  # hours
for k, ax in enumerate(axes):
    center = fcm.centers[k]
    dominant_days = np.argsort(-U[k])[:20]
    for d in dominant_days:
        ax.plot(slots, irrad[d], color='gray', alpha=0.2, lw=0.8)
    ax.plot(slots, center, color='crimson', lw=2, label='centroid')
    ax.set_title(f'Regime {k+1}  (mean membership={U[k].mean():.2f})')
    ax.set_xlabel('Hour')
axes[0].set_ylabel('NWP global irradiance')
fig.suptitle(f'{sid} — FCM weather regimes (K={K})', fontsize=13)
plt.tight_layout()
plt.show()

## 9. VMD leakage check (train/test irradiance overlap)
The `.npy` files stored alongside each station CSV contain the VMD decomposition used **only on training data** to prevent leakage.

In [ ]:
sid = STATIONS[0]
vmd = np.load(DATA_DIR / f'{sid}_vmd_leak.npy')  # shape (n_modes, T)
print(f'VMD modes shape: {vmd.shape}')

fig, axes = plt.subplots(vmd.shape[0], 1, figsize=(14, 2.5 * vmd.shape[0]), sharex=True)
t = np.arange(vmd.shape[1])
for i, ax in enumerate(axes):
    ax.plot(t[:96*30], vmd[i, :96*30], lw=0.8)  # first 30 days
    ax.set_ylabel(f'IMF {i+1}')
axes[-1].set_xlabel('Time step (15-min)')
fig.suptitle(f'{sid} — VMD intrinsic mode functions (first 30 days)', fontsize=12)
plt.tight_layout()
plt.show()